# 📊 Análisis de Fraude — Transferencias a Terceros
**Scotiabank Perú · Prevención de Fraude**  
**Criterio Monitor:** COD_TRX=40 · TIPO_CUENTA=1010 · MONTO ≥ S/1,200

---
> **Indicadores:** `F`=Fraude · `G`=Buena · `P`=Pendiente · `D`=Descarte · `N`=Normal

**Stack:** `pandas` (solo lectura Excel) · `polars` (todo el procesamiento) · `plotly` (gráficos interactivos) · `itables` (tablas DT estilo R)

## 0. Setup — Paquetes, tema y tablas interactivas

In [ ]:
import subprocess, sys
for pkg in ['polars','pandas','numpy','plotly','itables','scikit-learn',
            'networkx','pyarrow','openpyxl','lightgbm']:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

import polars as pl
import polars.selectors as cs
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx
from pathlib import Path
import re, time, warnings
warnings.filterwarnings('ignore')

# Tablas interactivas estilo DT de R
from itables import init_notebook_mode, show
import itables.options as opt
init_notebook_mode(connected=True)
opt.lengthMenu = [10, 25, 50, 100, -1]
opt.classes    = 'display nowrap compact hover stripe'
opt.style      = 'table-layout:auto;width:100%;font-size:12px'
opt.maxBytes   = 0
opt.scrollX    = True

# Paleta de colores institucional
C_FRAUD  = '#c0392b'
C_OK     = '#2980b9'
C_ACCENT = '#e67e22'
C_MUTED  = '#7f8c8d'
C_GOOD   = '#27ae60'

# Tema Plotly global
import plotly.io as pio
pio.templates.default = 'plotly_white'

print(f'Polars  {pl.__version__}')
print(f'Pandas  {pd.__version__}')
print('✅ Setup completo')

---
# 1. Ingesta — Lectura con pandas, procesamiento con Polars

In [ ]:
BASE_DIR = Path(r'C:\Users\s4930359\Data_Herramientas\data\transferencias_terceros')

ARCHIVOS = {
    'Enero' : BASE_DIR / 'monitor_transferencias_enero.xlsx',
    'Marzo' : BASE_DIR / 'monitor_transferencias_marzo.xlsx',
    'Abril' : BASE_DIR / 'monitor_transferencias_abril.xlsx',
}

def limpiar_nombre(n: str) -> str:
    n = n.strip().upper()
    n = re.sub(r'\s+', '_', n)
    n = re.sub(r'-',   '_', n)
    n = re.sub(r'[^A-Z0-9_]', '', n)
    return n

def leer_monitor(path: Path, mes: str) -> pl.DataFrame:
    """Pandas lee el Excel (estable) → conversión inmediata a Polars."""
    df_pd = pd.read_excel(path, skiprows=4, dtype=str, header=0, engine='openpyxl')
    df_pd.columns = [limpiar_nombre(c) for c in df_pd.columns]
    return pl.from_pandas(df_pd).with_columns(pl.lit(mes).alias('MES_ORIGEN'))

t0 = time.time()
frames = [leer_monitor(p, m) for m, p in ARCHIVOS.items()]
raw    = pl.concat(frames, how='diagonal_relaxed')
print(f'⏱️  Ingesta: {time.time()-t0:.1f}s')
print(f'📊 Shape:    {raw.shape[0]:,} filas × {raw.shape[1]} columnas')
print(f'💾 Memoria:  {raw.estimated_size("mb"):.1f} MB')

## 1.1 Inspección de columnas

In [ ]:
for i, c in enumerate(raw.columns, 1):
    print(f'{i:>3}. {c}')

## 1.2 Mapeo de columnas

> ⚠️ Completa la **clave izquierda** con el nombre exacto que apareció arriba.

In [ ]:
COL_MAP = {
    # Nombre exacto en Monitor              : Alias interno
    'NOMBRE_EXACTO_EN_MONITOR'              : 'FECHA_TRX',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'HORA_TRX',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'MONTO',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'INDICADOR',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'ALERTA',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'CUENTA',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'ID_CLIENTE',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'PRIMER_NOMBRE_CLIENTE',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'APELLIDO_CLIENTE',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'NOMBRE_BENEFICIARIO',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'CUENTA_DESTINO',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'HASH_DISPOSITIVO',
    'NOMBRE_EXACTO_EN_MONITOR'              : 'CANAL',
    'ACFTARJETA_RESGISTRO_750'              : 'TARJETA_REG',
    'ACFTARJETA_POS_76_DIGITOS'             : 'TARJETA_MID',
    'ACFSALDO_DISPONIBLE_EN_MONEDA_TRX'     : 'SALDO_DISPONIBLE_RAW',
    'CC_K05_COUNTTMP_TAMANO_COMERCIO'       : 'CNT_MES_PREVIO',
}

cols_ok  = {k:v for k,v in COL_MAP.items() if k in raw.columns}
cols_err = [k for k in COL_MAP if k not in raw.columns]
df = raw.rename(cols_ok)
if cols_err:
    print(f'⚠️  No encontradas: {cols_err}')
else:
    print(f'✅ Mapeadas {len(cols_ok)} columnas')

## 1.3 Casteo de tipos

In [ ]:
df = (
    df
    # Numéricos
    .with_columns([
        pl.col('MONTO').cast(pl.Float64, strict=False),
        pl.col('SALDO_DISPONIBLE_RAW').cast(pl.Float64, strict=False).alias('SALDO_DISPONIBLE'),
        pl.col('CNT_MES_PREVIO').cast(pl.Int32, strict=False),
    ])
    .drop('SALDO_DISPONIBLE_RAW')
    # Fecha (AAAAMMDD pegado)
    .with_columns(
        pl.col('FECHA_TRX').str.strip_chars().str.to_date('%Y%m%d', strict=False)
    )
    # Hora (HH:MM:SS con dos puntos)
    .with_columns(
        pl.col('HORA_TRX').str.strip_chars().str.to_time('%H:%M:%S', strict=False)
    )
    # Datetime combinado
    .with_columns(
        (pl.col('FECHA_TRX').cast(pl.Utf8) + ' ' + pl.col('HORA_TRX').cast(pl.Utf8))
          .str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S', strict=False)
          .alias('FECHA_HORA')
    )
    # Tarjeta reconstruida
    .with_columns([
        (pl.col('TARJETA_REG').str.strip_chars().str.slice(0,6) +
         pl.col('TARJETA_MID').str.strip_chars() +
         pl.col('TARJETA_REG').str.strip_chars().str.slice(-4)).alias('TARJETA_FULL'),
        pl.col('TARJETA_REG').str.strip_chars().str.slice(0,6).alias('BIN'),
        pl.col('TARJETA_REG').str.strip_chars().str.slice(-4).alias('TARJETA_LAST4'),
    ])
)

# Limpiar strings clave
for c in ['INDICADOR','CANAL','NOMBRE_BENEFICIARIO','HASH_DISPOSITIVO']:
    if c in df.columns:
        df = df.with_columns(pl.col(c).str.strip_chars().str.to_uppercase())

print(f'✅ Casteo completo: {df.shape[0]:,} × {df.shape[1]}')
show(df.select(['FECHA_TRX','HORA_TRX','MONTO','SALDO_DISPONIBLE',
                'CNT_MES_PREVIO','TARJETA_FULL','BIN','CANAL','INDICADOR']).head(10).to_pandas())

---
# 2. Ingeniería de Variables

## 2.1 Flags básicos: objetivo, temporales, monto, saldo, historia

In [ ]:
df = df.with_columns([
    # ── Objetivo ──────────────────────────────────────────────
    (pl.col('INDICADOR')=='F').cast(pl.Int8).alias('FLAG_FRAUDE'),
    pl.col('INDICADOR').is_in(['F','G','D']).cast(pl.Int8).alias('FLAG_CLASIFICADO'),
    (pl.col('INDICADOR')=='P').cast(pl.Int8).alias('FLAG_PENDIENTE'),

    # ── Temporales ────────────────────────────────────────────
    pl.col('FECHA_TRX').dt.month().alias('MES'),
    pl.col('FECHA_TRX').dt.day().alias('DIA'),
    pl.col('FECHA_TRX').dt.weekday().alias('DIA_SEMANA'),
    pl.col('FECHA_HORA').dt.hour().alias('HORA'),
    pl.when(pl.col('FECHA_HORA').dt.hour().is_between(0,5)).then(pl.lit('Madrugada'))
      .when(pl.col('FECHA_HORA').dt.hour().is_between(6,11)).then(pl.lit('Mañana'))
      .when(pl.col('FECHA_HORA').dt.hour().is_between(12,17)).then(pl.lit('Tarde'))
      .otherwise(pl.lit('Noche')).alias('FRANJA_HORARIA'),
    (pl.col('FECHA_TRX').dt.weekday()>=6).cast(pl.Int8).alias('FLAG_FIN_SEMANA'),
    (pl.col('FECHA_TRX').dt.day()>=26).cast(pl.Int8).alias('FLAG_FIN_MES'),

    # ── Monto ─────────────────────────────────────────────────
    (pl.col('MONTO')>=5000).cast(pl.Int8).alias('FLAG_MONTO_ALTO'),
    (pl.col('MONTO')>=10000).cast(pl.Int8).alias('FLAG_MONTO_CRITICO'),
])

# Saldo y ratio
df = df.with_columns(
    (pl.col('MONTO') / (pl.col('SALDO_DISPONIBLE')+1e-9)).round(4).alias('RATIO_MONTO_SALDO')
).with_columns([
    (pl.col('RATIO_MONTO_SALDO')>=0.80).cast(pl.Int8).alias('FLAG_VACIADO_CUENTA'),
    (pl.col('SALDO_DISPONIBLE')<200).cast(pl.Int8).alias('FLAG_SALDO_BAJO'),
    # Historia de cuenta
    (pl.col('CNT_MES_PREVIO').is_null() | (pl.col('CNT_MES_PREVIO')==0))
      .cast(pl.Int8).alias('FLAG_SIN_HISTORIA'),
    (pl.col('CNT_MES_PREVIO')<=3).cast(pl.Int8).alias('FLAG_CUENTA_NUEVA'),
    # Beneficiario
    (pl.col('NOMBRE_BENEFICIARIO').is_null() |
     (pl.col('NOMBRE_BENEFICIARIO').str.len_chars()==0))
      .cast(pl.Int8).alias('FLAG_BENE_NULO'),
])

print('✅ Flags básicos creados')

## 2.2 🚀 Velocidad por ventanas de tiempo (lo nuevo importante)

**Ventanas calculadas por cliente, ordenadas por FECHA_HORA:**
- 3 minutos · 5 minutos · 10 minutos
- 1 hora · 6 horas · 12 horas · 24 horas
- 7 días

Para cada ventana se calcula: **N° transacciones**, **monto total**, **N° fraudes**, **monto fraudulento**.

In [ ]:
# Ordenamiento crítico: por cliente y tiempo
df = df.sort(['CUENTA','FECHA_HORA'])

VENTANAS = {
    '3MIN' : '3m',
    '5MIN' : '5m',
    '10MIN': '10m',
    '1H'   : '1h',
    '6H'   : '6h',
    '12H'  : '12h',
    '24H'  : '24h',
    '7D'   : '7d',
}

# Calcular rolling agregaciones por cuenta
for nombre, ventana in VENTANAS.items():
    df = df.with_columns([
        # N° transacciones en la ventana (incluye la actual)
        pl.len().rolling(
            index_column='FECHA_HORA', period=ventana, closed='right'
        ).over('CUENTA').alias(f'VEL_TRX_{nombre}'),
        # Monto acumulado en la ventana
        pl.col('MONTO').rolling_sum_by(
            by='FECHA_HORA', window_size=ventana, closed='right'
        ).over('CUENTA').alias(f'VEL_MONTO_{nombre}'),
        # N° fraudes en la ventana
        pl.col('FLAG_FRAUDE').rolling_sum_by(
            by='FECHA_HORA', window_size=ventana, closed='right'
        ).over('CUENTA').alias(f'VEL_FRAUDES_{nombre}'),
    ])

print('✅ Ventanas de velocidad por cliente creadas:')
for n in VENTANAS:
    print(f'   · VEL_TRX_{n}, VEL_MONTO_{n}, VEL_FRAUDES_{n}')

# Flags de velocidad alta
df = df.with_columns([
    (pl.col('VEL_TRX_5MIN') >= 3).cast(pl.Int8).alias('FLAG_VEL_RAFAGA_5MIN'),     # 3+ trx en 5 min
    (pl.col('VEL_TRX_1H')   >= 5).cast(pl.Int8).alias('FLAG_VEL_ALTA_1H'),
    (pl.col('VEL_TRX_24H')  >= 10).cast(pl.Int8).alias('FLAG_VEL_ALTA_24H'),
    (pl.col('VEL_TRX_7D')   >= 30).cast(pl.Int8).alias('FLAG_VEL_ALTA_7D'),
    # Compatibilidad con código viejo
    (pl.col('VEL_TRX_24H')  >  3).cast(pl.Int8).alias('FLAG_VEL_ALTA'),
])

print('\n✅ Flags de velocidad creados')

## 2.3 Concentración beneficiario, ATO dispositivo, Z-score

In [ ]:
# Concentración beneficiario
conc_bene = (
    df.filter(pl.col('FLAG_BENE_NULO')==0)
      .group_by('NOMBRE_BENEFICIARIO')
      .agg(pl.col('CUENTA').n_unique().alias('N_CUENTAS_ORIGEN_AL_BENE'))
)
df = df.join(conc_bene, on='NOMBRE_BENEFICIARIO', how='left').with_columns(
    (pl.col('N_CUENTAS_ORIGEN_AL_BENE')>=5).cast(pl.Int8).alias('FLAG_BENE_CONCENTRADO')
)

# ATO dispositivo
hash_multi = (
    df.filter(pl.col('HASH_DISPOSITIVO').is_not_null())
      .group_by('HASH_DISPOSITIVO')
      .agg(pl.col('CUENTA').n_unique().alias('N_CUENTAS_POR_DISPOSITIVO'))
)
df = df.join(hash_multi, on='HASH_DISPOSITIVO', how='left').with_columns(
    (pl.col('N_CUENTAS_POR_DISPOSITIVO')>=3).cast(pl.Int8).alias('FLAG_DISPOSITIVO_MULTIUSUARIO')
)

# Z-score de monto por cuenta
stats = df.group_by('CUENTA').agg([
    pl.col('MONTO').mean().alias('_mu'), pl.col('MONTO').std().alias('_sd')
])
df = (
    df.join(stats, on='CUENTA', how='left')
      .with_columns(
          ((pl.col('MONTO')-pl.col('_mu'))/(pl.col('_sd')+1e-9)).round(3).alias('ZSCORE_MONTO_CUENTA')
      ).drop(['_mu','_sd'])
)
df = df.with_columns(
    (pl.col('ZSCORE_MONTO_CUENTA').abs()>2).cast(pl.Int8).alias('FLAG_ANOMALIA_MONTO')
)

# ── Combinaciones inteligentes ────────────────────────────────────────────────
df = df.with_columns([
    # Anomalía + ráfaga: monto inusual + muchas trx en 5 min
    ((pl.col('FLAG_ANOMALIA_MONTO')==1) & (pl.col('FLAG_VEL_RAFAGA_5MIN')==1))
      .cast(pl.Int8).alias('FLAG_ANOMALIA_RAFAGA'),
    # Vaciado + cuenta nueva: alta sospecha de cuenta mula
    ((pl.col('FLAG_VACIADO_CUENTA')==1) & (pl.col('FLAG_CUENTA_NUEVA')==1))
      .cast(pl.Int8).alias('FLAG_MULA'),
    # ATO + monto alto + madrugada
    ((pl.col('FLAG_DISPOSITIVO_MULTIUSUARIO')==1) &
     (pl.col('FLAG_MONTO_ALTO')==1) &
     (pl.col('FRANJA_HORARIA')=='Madrugada')).cast(pl.Int8).alias('FLAG_ATO_NOCTURNO'),
])

# Datasets derivados
df_cal = df.filter(pl.col('FLAG_CLASIFICADO')==1)
df_f   = df.filter(pl.col('INDICADOR')=='F')

print(f'df       : {df.shape[0]:>10,}  (todo)')
print(f'df_cal   : {df_cal.shape[0]:>10,}  (clasificadas F+G+D)')
print(f'df_f     : {df_f.shape[0]:>10,}  (solo F)')
print(f'\nFlags totales: {len([c for c in df.columns if c.startswith("FLAG_")])}')

---
# 3. KPIs y Métricas Ejecutivas

**Métricas que importan al especialista:**
- **Fraud Rate** → probabilidad de fraude del segmento
- **Severidad** → monto promedio por caso de fraude
- **LIFT** → cuánto más fraude tiene un segmento vs el promedio (>1 = más sospechoso)
- **EFL** (Expected Fraud Loss) → pérdida esperada = P(fraude) × monto

In [ ]:
total_trx   = df.shape[0]
total_cal   = df_cal.shape[0]
total_fraud = df_f.shape[0]
monto_fraud = df_f['MONTO'].sum()
fraud_rate  = total_fraud / total_cal * 100 if total_cal else 0
severidad   = monto_fraud / total_fraud if total_fraud else 0
tasa_global = total_fraud / total_cal if total_cal else 0

kpis = pd.DataFrame({
    'KPI'   : ['Total transacciones','Clasificadas (F+G+D)','Fraudes (F)',
               'Monto fraude (S/)','Fraud Rate (%)','Severidad (S/)'],
    'Valor' : [f'{total_trx:,}', f'{total_cal:,}', f'{total_fraud:,}',
               f'{monto_fraud:,.2f}', f'{fraud_rate:.4f}',
               f'{severidad:,.2f}']
})
show(kpis)

---
# 4. Análisis Univariado — Variables Continuas

## 4.1 MONTO — Distribución interactiva (Plotly con zoom y hover)

In [ ]:
df_monto = (
    df_cal.filter(pl.col('MONTO').is_not_null())
          .with_columns(
              pl.when(pl.col('FLAG_FRAUDE')==1).then(pl.lit('Fraude'))
                .otherwise(pl.lit('No Fraude')).alias('GRUPO')
          )
          .select(['MONTO','GRUPO','INDICADOR'])
          .to_pandas()
)

# Histograma + densidad por grupo
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Densidad superpuesta','Boxplot comparativo'))

for grupo, color in [('No Fraude', C_OK), ('Fraude', C_FRAUD)]:
    datos = df_monto[df_monto['GRUPO']==grupo]['MONTO']
    fig.add_trace(go.Histogram(
        x=datos, name=grupo, marker_color=color, opacity=0.6,
        histnorm='probability density', nbinsx=50,
        hovertemplate='Monto: %{x:,.0f}<br>Densidad: %{y:.4f}<extra></extra>'
    ), row=1, col=1)

    fig.add_trace(go.Box(
        y=datos, name=grupo, marker_color=color, boxmean='sd',
        hovertemplate='%{y:,.0f}<extra></extra>'
    ), row=1, col=2)

fig.update_layout(
    title='Distribución de Monto — Fraude vs No Fraude (zoom + hover)',
    height=400, barmode='overlay', showlegend=True
)
fig.update_xaxes(title='Monto (S/)', row=1, col=1)
fig.update_yaxes(title='Densidad', row=1, col=1)
fig.update_yaxes(title='Monto (S/)', row=1, col=2)
fig.show()

## 4.2 MONTO — Estadísticas por indicador

In [ ]:
stats_monto = (
    df_cal.filter(pl.col('MONTO').is_not_null())
          .group_by('INDICADOR')
          .agg([
              pl.len().alias('N'),
              pl.col('MONTO').min().round(2).alias('Min'),
              pl.col('MONTO').quantile(0.25).round(2).alias('P25'),
              pl.col('MONTO').median().round(2).alias('Mediana'),
              pl.col('MONTO').mean().round(2).alias('Media'),
              pl.col('MONTO').quantile(0.75).round(2).alias('P75'),
              pl.col('MONTO').quantile(0.90).round(2).alias('P90'),
              pl.col('MONTO').max().round(2).alias('Max'),
              pl.col('MONTO').std().round(2).alias('SD'),
          ]).sort('INDICADOR')
).to_pandas()
show(stats_monto, caption='Estadísticas de Monto por Indicador')

## 4.3 Discretización de MONTO — 3 métodos comparados

**Métodos:**
1. **Boxplot/Cuartiles** — corta por percentiles, NO usa el target
2. **Árbol CART ✅** — corta donde mejor se separa fraude vs no fraude (USA target)
3. **K-means cluster** — agrupa por similitud de valores, NO usa target

El **CART es el más relevante para detección de fraude** porque optimiza la separación.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans

disc = (
    df_cal.filter(pl.col('MONTO').is_not_null() & pl.col('FLAG_FRAUDE').is_not_null())
          .select(['MONTO','FLAG_FRAUDE'])
          .to_pandas()
)

# ── Método 1: Cuartiles ──────────────────────────────────────────────────────
q = disc['MONTO'].quantile([0.25, 0.5, 0.75]).tolist()
disc['CAT_CUARTIL'] = pd.cut(disc['MONTO'],
    bins=[-np.inf]+q+[np.inf],
    labels=['Q1 Bajo','Q2 Med-Bajo','Q3 Med-Alto','Q4 Alto'])

# ── Método 2: Árbol CART ─────────────────────────────────────────────────────
arbol = DecisionTreeClassifier(
    max_depth=3, min_samples_leaf=10, class_weight='balanced'
)
arbol.fit(disc[['MONTO']].values, disc['FLAG_FRAUDE'].values)

cortes_cart = sorted(set(arbol.tree_.threshold[arbol.tree_.threshold!=-2]))
print(f'📋 Cortes CART: {[round(c,2) for c in cortes_cart]}')

breaks = [-np.inf] + cortes_cart + [np.inf]
labels_cart = [f'S/{breaks[i]:,.0f}-{breaks[i+1]:,.0f}' for i in range(len(breaks)-1)]
labels_cart[-1] = f'S/{cortes_cart[-1]:,.0f}+'
disc['CAT_CART'] = pd.cut(disc['MONTO'], bins=breaks, labels=labels_cart, right=False)

# ── Método 3: K-means ────────────────────────────────────────────────────────
km = KMeans(n_clusters=4, random_state=42, n_init=10)
disc['CAT_KMEANS_RAW'] = km.fit_predict(disc[['MONTO']].values)
# Ordenar clusters por monto promedio
orden_km = (disc.groupby('CAT_KMEANS_RAW')['MONTO'].mean()
                  .sort_values().reset_index())
mapa_km = {row['CAT_KMEANS_RAW']: f'K{i+1}' for i, row in orden_km.iterrows()}
disc['CAT_KMEANS'] = disc['CAT_KMEANS_RAW'].map(mapa_km)

In [ ]:
# Función helper para tabla de proporciones
def resumen_disc(data, col):
    tab = (
        data.groupby(col, observed=True)
            .agg(TOTAL=('FLAG_FRAUDE','size'),
                 FRAUDES=('FLAG_FRAUDE','sum'))
            .reset_index()
    )
    tab['FRAUD_RATE'] = (tab['FRAUDES']/tab['TOTAL']*100).round(3)
    tab['LIFT']       = (tab['FRAUDES']/tab['TOTAL'] / (disc['FLAG_FRAUDE'].mean())).round(2)
    return tab

fig = make_subplots(rows=1, cols=3,
    subplot_titles=('Cuartiles','Árbol CART ✅ Recomendado','K-means'))

for i, (col, titulo) in enumerate([('CAT_CUARTIL','Cuartiles'),
                                     ('CAT_CART','CART'),
                                     ('CAT_KMEANS','K-means')], start=1):
    tab = resumen_disc(disc, col)
    fig.add_trace(go.Bar(
        x=tab[col].astype(str), y=tab['FRAUD_RATE'],
        marker_color=C_FRAUD, name=titulo, showlegend=False,
        text=[f'{v}%' for v in tab['FRAUD_RATE']],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Fraud Rate: %{y}%<br>Total: %{customdata[0]:,}<br>Fraudes: %{customdata[1]}<extra></extra>',
        customdata=tab[['TOTAL','FRAUDES']].values
    ), row=1, col=i)
    fig.update_xaxes(tickangle=-30, row=1, col=i)

fig.update_layout(title='Comparación de 3 Métodos de Discretización — Fraud Rate',
                  height=420)
fig.update_yaxes(title='Fraud Rate (%)', row=1, col=1)
fig.show()

## 4.4 Árbol CART interactivo + Scatter de partición

In [ ]:
# ── Visualización del árbol ──────────────────────────────────────────────────
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

fig_arbol, ax = plt.subplots(figsize=(14, 5))
plot_tree(arbol, feature_names=['MONTO'], class_names=['No Fraude','Fraude'],
          filled=True, rounded=True, fontsize=10, ax=ax,
          impurity=False, proportion=True)
ax.set_title('Árbol CART — Cómo se parten los montos para discriminar fraude',
             fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

# ── Scatter de partición (estilo R) ──────────────────────────────────────────
# Muestra todos los puntos clasificados con líneas verticales en los cortes
disc_sample = disc.sample(n=min(5000, len(disc)), random_state=42)
disc_sample['JITTER'] = np.random.uniform(-0.4, 0.4, len(disc_sample))
disc_sample['Y_GRUPO'] = disc_sample['FLAG_FRAUDE'] + disc_sample['JITTER']
disc_sample['GRUPO_LABEL'] = disc_sample['FLAG_FRAUDE'].map({0:'No Fraude', 1:'Fraude'})

fig_scatter = px.scatter(
    disc_sample, x='MONTO', y='Y_GRUPO', color='GRUPO_LABEL',
    color_discrete_map={'No Fraude':C_OK,'Fraude':C_FRAUD},
    opacity=0.5, height=420,
    title='Scatter de Partición — Cómo el árbol corta el MONTO',
    labels={'Y_GRUPO':'Grupo','MONTO':'Monto (S/)','GRUPO_LABEL':''}
)
# Líneas verticales en los cortes
for c in cortes_cart:
    fig_scatter.add_vline(x=c, line_dash='dash', line_color=C_ACCENT,
                          line_width=2,
                          annotation_text=f'Corte: S/{c:,.0f}',
                          annotation_position='top')
fig_scatter.update_yaxes(tickvals=[0,1], ticktext=['No Fraude','Fraude'])
fig_scatter.show()

## 4.5 Evaluación REAL de la regla sobre TODO el volumen (incluyendo N)

**Importante:** los cortes del árbol vienen de las 430 clasificadas, pero la regla en producción se aplica sobre las 193k. Aquí vemos el costo operativo real.

In [ ]:
df = df.with_columns(
    pl.col('MONTO').cut(
        breaks=cortes_cart, labels=labels_cart, left_closed=True
    ).alias('MONTO_CAT_REGLA')
)

eval_real = (
    df.group_by('MONTO_CAT_REGLA')
      .agg([
          pl.len().alias('TOTAL'),
          (pl.col('INDICADOR')=='F').sum().alias('FRAUDES_F'),
          (pl.col('INDICADOR')=='G').sum().alias('BUENOS_G'),
          (pl.col('INDICADOR')=='N').sum().alias('NORMALES_N'),
          (pl.col('INDICADOR')=='P').sum().alias('PENDIENTES_P'),
          pl.col('MONTO').sum().round(2).alias('MONTO_TOTAL'),
          pl.col('MONTO').filter(pl.col('INDICADOR')=='F').sum().round(2).alias('MONTO_FRAUDE'),
      ])
      .with_columns([
          (pl.col('TOTAL')/df.shape[0]*100).round(2).alias('PCT_VOLUMEN'),
          (pl.col('FRAUDES_F')/pl.col('TOTAL')*100).round(4).alias('FRAUD_RATE'),
          (pl.col('FRAUDES_F')/total_fraud*100).round(2).alias('PCT_FRAUDE_CAP'),
      ])
      .sort('MONTO_CAT_REGLA')
).to_pandas()

show(eval_real, caption='⚠️ Impacto REAL — incluye N (las normales que la regla alertaría)')

---
# 5. Análisis Univariado — Categóricas

## 5.1 Indicador de Fraude

In [ ]:
tab_ind = (
    df.group_by('INDICADOR')
      .agg([
          pl.len().alias('N'),
          pl.col('MONTO').sum().round(2).alias('MONTO_TOTAL'),
          pl.col('MONTO').mean().round(2).alias('MONTO_PROM')
      ])
      .with_columns((pl.col('N')/df.shape[0]*100).round(2).alias('PCT'))
      .sort('N', descending=True)
).to_pandas()

fig = px.bar(tab_ind, x='INDICADOR', y='N', color='INDICADOR',
             color_discrete_map={'F':C_FRAUD,'G':C_OK,'N':C_MUTED,'P':C_ACCENT,'D':'#8e44ad'},
             text='PCT',
             title='Distribución por Indicador de Fraude')
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=380, showlegend=False)
fig.show()
show(tab_ind)

## 5.2 Canal × Franja Horaria — Heatmap

In [ ]:
canal_franja = (
    df_cal.group_by(['CANAL','FRANJA_HORARIA'])
          .agg([pl.len().alias('TOTAL'), pl.col('FLAG_FRAUDE').sum().alias('FRAUDES')])
          .with_columns((pl.col('FRAUDES')/pl.col('TOTAL')*100).round(3).alias('FRAUD_RATE'))
).to_pandas()

orden_franja = ['Madrugada','Mañana','Tarde','Noche']
pivot = canal_franja.pivot(index='CANAL', columns='FRANJA_HORARIA', values='FRAUD_RATE').reindex(columns=orden_franja)

fig = px.imshow(pivot, color_continuous_scale='Reds',
                aspect='auto', text_auto='.3f',
                title='Fraud Rate (%) — Canal × Franja Horaria',
                labels={'color':'Fraud Rate %'})
fig.update_layout(height=300)
fig.show()

## 5.3 Franja Horaria con LIFT y Severidad

In [ ]:
tab_franja = (
    df_cal.group_by('FRANJA_HORARIA').agg([
        pl.len().alias('TOTAL'),
        pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
        pl.col('MONTO').filter(pl.col('FLAG_FRAUDE')==1).sum().round(2).alias('MONTO_FRAUDE'),
    ]).with_columns([
        (pl.col('FRAUDES')/pl.col('TOTAL')*100).round(3).alias('FRAUD_RATE'),
        ((pl.col('FRAUDES')/pl.col('TOTAL'))/tasa_global).round(2).alias('LIFT'),
        (pl.col('MONTO_FRAUDE')/pl.col('FRAUDES')).round(2).alias('SEVERIDAD'),
    ]).sort('FRAUD_RATE', descending=True)
).to_pandas()
show(tab_franja, caption='LIFT > 1 = más fraude que el promedio')

---
# 6. Análisis Temporal — Evolución y Patrones de Hora

In [ ]:
evol = (
    df_cal.group_by('MES_ORIGEN').agg([
        pl.len().alias('TOTAL_TRX'),
        pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
        pl.col('MONTO').filter(pl.col('FLAG_FRAUDE')==1).sum().round(2).alias('MONTO_FRAUDE'),
    ]).with_columns([
        (pl.col('FRAUDES')/pl.col('TOTAL_TRX')*100).round(4).alias('FRAUD_RATE'),
        (pl.col('MONTO_FRAUDE')/pl.col('FRAUDES')).round(2).alias('SEVERIDAD'),
    ]).sort('MES_ORIGEN')
).to_pandas()

fig = make_subplots(specs=[[{'secondary_y':True}]])
fig.add_trace(go.Bar(x=evol['MES_ORIGEN'], y=evol['FRAUDES'],
                     marker_color=C_FRAUD, name='# Fraudes', opacity=0.8),
              secondary_y=False)
fig.add_trace(go.Scatter(x=evol['MES_ORIGEN'], y=evol['FRAUD_RATE'],
                         mode='lines+markers+text', line=dict(color=C_ACCENT, width=3),
                         marker=dict(size=10), text=evol['FRAUD_RATE'].round(3),
                         textposition='top center', name='Fraud Rate %'),
              secondary_y=True)
fig.update_layout(title='Evolución Mensual — Fraudes y Fraud Rate', height=400)
fig.update_yaxes(title='# Fraudes', secondary_y=False)
fig.update_yaxes(title='Fraud Rate (%)', secondary_y=True)
fig.show()
show(evol)

In [ ]:
por_hora = (
    df_cal.group_by('HORA').agg([
        pl.len().alias('TOTAL'), pl.col('FLAG_FRAUDE').sum().alias('FRAUDES')
    ]).with_columns((pl.col('FRAUDES')/pl.col('TOTAL')*100).round(3).alias('FRAUD_RATE'))
      .sort('HORA')
).to_pandas()

fig = make_subplots(specs=[[{'secondary_y':True}]])
fig.add_trace(go.Bar(x=por_hora['HORA'], y=por_hora['TOTAL'],
                     marker_color=C_OK, opacity=0.5, name='Volumen'),
              secondary_y=False)
fig.add_trace(go.Scatter(x=por_hora['HORA'], y=por_hora['FRAUD_RATE'],
                         mode='lines+markers', line=dict(color=C_FRAUD, width=3),
                         marker=dict(size=8), name='Fraud Rate %'),
              secondary_y=True)
fig.update_layout(title='Volumen vs Fraud Rate por Hora del Día',
                  xaxis=dict(tickmode='linear', dtick=1), height=400)
fig.update_yaxes(title='Volumen', secondary_y=False)
fig.update_yaxes(title='Fraud Rate (%)', secondary_y=True)
fig.show()

---
# 7. Análisis de Velocidad por Ventanas

**Verificación:** ¿Las cuentas con muchas trx en ventanas cortas tienen más fraude?

In [ ]:
# Comparar fraud rate por nivel de velocidad en cada ventana
ventanas_cols = [(f'VEL_TRX_{n}', n) for n in VENTANAS]

resumen_vel = []
for col, ventana in ventanas_cols:
    if col not in df_cal.columns: continue
    # Discretizar la velocidad en 4 niveles
    vals = df_cal.select(col).to_pandas()[col].dropna()
    if vals.nunique() < 2: continue
    qs = vals.quantile([0.5, 0.9, 0.99]).tolist()
    qs = sorted(set([1] + [int(q) for q in qs]))

    df_t = df_cal.with_columns(
        pl.col(col).cut(breaks=qs, labels=None).alias('NIVEL')
    )
    res = (
        df_t.group_by('NIVEL').agg([
            pl.len().alias('TOTAL'),
            pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
        ]).with_columns([
            (pl.col('FRAUDES')/pl.col('TOTAL')*100).round(3).alias('FRAUD_RATE'),
            pl.lit(ventana).alias('VENTANA'),
        ])
    ).to_pandas()
    resumen_vel.append(res)

if resumen_vel:
    df_vel = pd.concat(resumen_vel, ignore_index=True)
    df_vel['NIVEL'] = df_vel['NIVEL'].astype(str)

    fig = px.bar(df_vel, x='NIVEL', y='FRAUD_RATE', color='VENTANA',
                 facet_col='VENTANA', facet_col_wrap=4,
                 title='Fraud Rate por Nivel de Velocidad — cada ventana',
                 height=500)
    fig.update_layout(showlegend=False)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig.show()

## 7.1 Top cuentas con velocidad alta

In [ ]:
vel_cuenta = (
    df.group_by('CUENTA').agg([
        pl.len().alias('TOTAL_TRX'),
        pl.col('FLAG_FRAUDE').sum().alias('FRAUDES'),
        (pl.col('INDICADOR')=='G').sum().alias('BUENOS_G'),
        (pl.col('INDICADOR')=='N').sum().alias('NORMALES_N'),
        (pl.col('INDICADOR')=='P').sum().alias('PENDIENTES_P'),
        pl.col('MONTO').sum().round(2).alias('MONTO_TOTAL'),
        pl.col('MONTO').filter(pl.col('FLAG_FRAUDE')==1).sum().round(2).alias('MONTO_FRAUDE'),
        pl.col('FECHA_TRX').n_unique().alias('DIAS_ACTIVO'),
        pl.col('CUENTA_DESTINO').n_unique().alias('N_DESTINOS_DISTINTOS'),
        pl.col('VEL_TRX_5MIN').max().alias('MAX_VEL_5MIN'),
        pl.col('VEL_TRX_24H').max().alias('MAX_VEL_24H'),
    ]).with_columns([
        (pl.col('FRAUDES')/pl.col('TOTAL_TRX')*100).round(3).alias('FRAUD_RATE'),
        (pl.col('TOTAL_TRX')/(pl.col('DIAS_ACTIVO')+1)).round(2).alias('TRX_POR_DIA'),
    ]).filter(pl.col('FRAUDES')>0)
      .sort('MONTO_FRAUDE', descending=True)
).to_pandas()

show(vel_cuenta.head(30), caption='Top 30 cuentas con fraude — ordenadas por monto')

## 7.2 Verificación interactiva — ver transacciones de cuentas sospechosas

Cambia `UMBRAL_TRX_DIA` para explorar diferentes niveles.

In [ ]:
UMBRAL_TRX_DIA = 3

cuentas_alta_vel = (
    vel_cuenta.query('TRX_POR_DIA > @UMBRAL_TRX_DIA & FRAUDES > 0')['CUENTA'].tolist()
)
print(f'Cuentas con >{UMBRAL_TRX_DIA} trx/día Y fraude: {len(cuentas_alta_vel)}')

trx_alta_vel = (
    df.filter(pl.col('CUENTA').is_in(cuentas_alta_vel))
      .select(['CUENTA','CUENTA_DESTINO','FECHA_TRX','HORA_TRX','MONTO',
               'SALDO_DISPONIBLE','INDICADOR','CANAL','NOMBRE_BENEFICIARIO',
               'VEL_TRX_5MIN','VEL_TRX_1H','VEL_TRX_24H','FLAG_FRAUDE'])
      .sort(['CUENTA','FECHA_TRX','HORA_TRX'])
).to_pandas()

show(trx_alta_vel, caption=f'Transacciones de cuentas con >{UMBRAL_TRX_DIA} trx/día y fraude')

---
# 8. Cuenta Destino — Top receptores de fraude

In [ ]:
top_destino = (
    df.filter((pl.col('INDICADOR')=='F') & pl.col('CUENTA_DESTINO').is_not_null())
      .group_by('CUENTA_DESTINO').agg([
          pl.len().alias('N_FRAUDES'),
          pl.col('CUENTA').n_unique().alias('N_CUENTAS_ORIGEN'),
          pl.col('NOMBRE_BENEFICIARIO').first().alias('NOMBRE_BENEFICIARIO'),
          pl.col('MONTO').sum().round(2).alias('MONTO_FRAUDE'),
          pl.col('FECHA_TRX').min().alias('PRIMERA_TRX'),
          pl.col('FECHA_TRX').max().alias('ULTIMA_TRX'),
      ]).sort('MONTO_FRAUDE', descending=True)
).to_pandas()

show(top_destino, caption='Top cuentas destino que reciben fraude — buscar/filtrar libremente')

---
# 9. Grafo de Red de Fraude — Beneficiarios concentrados

**Cómo jugar con el grafo:**
- **Zoom y pan**: con el mouse
- **Hover**: muestra cliente y monto
- **Toggle de leyenda**: clic para ocultar tipos
- **Reset**: doble clic

In [ ]:
edges_fraude = (
    df.filter((pl.col('INDICADOR')=='F') &
              pl.col('NOMBRE_BENEFICIARIO').is_not_null() &
              (pl.col('NOMBRE_BENEFICIARIO').str.len_chars()>0))
      .select([
          pl.col('CUENTA').alias('from'),
          pl.col('NOMBRE_BENEFICIARIO').alias('to'),
          pl.col('MONTO').alias('monto'),
          pl.col('PRIMER_NOMBRE_CLIENTE').alias('cliente_nombre'),
          pl.col('APELLIDO_CLIENTE').alias('cliente_apellido'),
          pl.col('FECHA_TRX'),
      ])
)

bene_multi = edges_fraude.group_by('to').agg(pl.col('from').n_unique().alias('n')).filter(pl.col('n')>=2)
edges_red  = edges_fraude.filter(pl.col('to').is_in(bene_multi['to'])).to_pandas()

if len(edges_red) > 0:
    G = nx.DiGraph()
    for _, row in edges_red.iterrows():
        nombre_cliente = f"{row['cliente_nombre'] or ''} {row['cliente_apellido'] or ''}".strip() or row['from']
        G.add_node(row['from'], tipo='Cuenta Origen', label=nombre_cliente)
        G.add_node(row['to'],   tipo='Beneficiario', label=row['to'])
        G.add_edge(row['from'], row['to'], weight=row['monto'])

    pos = nx.spring_layout(G, seed=42, k=2, iterations=50)

    edge_x, edge_y, edge_text = [], [], []
    for u, v, d in G.edges(data=True):
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]
        edge_text.append(f'{u} → {v}<br>S/{d["weight"]:,.0f}')

    nodes_origen_x   = [pos[n][0] for n in G.nodes if G.nodes[n].get('tipo')=='Cuenta Origen']
    nodes_origen_y   = [pos[n][1] for n in G.nodes if G.nodes[n].get('tipo')=='Cuenta Origen']
    nodes_origen_txt = [G.nodes[n].get('label','') for n in G.nodes if G.nodes[n].get('tipo')=='Cuenta Origen']

    nodes_bene_x   = [pos[n][0] for n in G.nodes if G.nodes[n].get('tipo')=='Beneficiario']
    nodes_bene_y   = [pos[n][1] for n in G.nodes if G.nodes[n].get('tipo')=='Beneficiario']
    nodes_bene_txt = [G.nodes[n].get('label','') for n in G.nodes if G.nodes[n].get('tipo')=='Beneficiario']
    nodes_bene_size= [G.in_degree(n)*8+10 for n in G.nodes if G.nodes[n].get('tipo')=='Beneficiario']

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
                              line=dict(color='#bdc3c7', width=1),
                              hoverinfo='none', name='Transferencias'))
    fig.add_trace(go.Scatter(x=nodes_origen_x, y=nodes_origen_y, mode='markers',
                              marker=dict(color=C_OK, size=12,
                                          line=dict(color='white', width=1)),
                              text=nodes_origen_txt, hoverinfo='text',
                              name='Cuenta Origen (cliente)'))
    fig.add_trace(go.Scatter(x=nodes_bene_x, y=nodes_bene_y, mode='markers+text',
                              marker=dict(color=C_FRAUD, size=nodes_bene_size,
                                          line=dict(color='white', width=2)),
                              text=nodes_bene_txt, textposition='top center',
                              textfont=dict(size=9, color='#333'),
                              hoverinfo='text', name='Beneficiario (recibe fraude)'))
    fig.update_layout(
        title='Red de Fraude — Beneficiarios desde múltiples cuentas (zoom + hover)',
        showlegend=True, height=650,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        hovermode='closest', plot_bgcolor='white'
    )
    fig.show()
else:
    print('ℹ️  No hay suficientes fraudes con beneficiario para el grafo')

---
# 10. Reglas Propuestas — Evaluación con Equilibrio

**Cómo leer la tabla:**
- **Precision** = de lo alertado, qué % es fraude (alto = pocos falsos positivos)
- **Recall** = del total de fraudes, qué % capturo
- **F1** = equilibrio entre Precision y Recall
- **Pct_Volumen** = qué % del volumen total alertaríamos (costo operativo)
- **Normales_N** = transacciones normales que se alertarían (ruido)

In [ ]:
def evaluar_regla(nombre, mascara):
    sub = df.filter(mascara)
    n_alertas = sub.shape[0]
    n_fraudes = sub.filter(pl.col('INDICADOR')=='F').shape[0]
    n_normales= sub.filter(pl.col('INDICADOR')=='N').shape[0]
    monto_cap = sub.filter(pl.col('INDICADOR')=='F')['MONTO'].sum()
    precision = n_fraudes/n_alertas*100 if n_alertas>0 else 0
    recall    = n_fraudes/total_fraud*100 if total_fraud>0 else 0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
    return {
        'Regla': nombre, 'Alertas': n_alertas, 'Fraudes_F': n_fraudes,
        'Normales_N': n_normales, 'Monto_Cap': round(monto_cap,2),
        'Precision': round(precision,2), 'Recall': round(recall,2),
        'F1': round(f1,2), 'Pct_Volumen': round(n_alertas/df.shape[0]*100,2),
    }

REGLAS = [
    ('R01: Monto Alto (≥5K)',                pl.col('MONTO')>=5000),
    ('R02: Monto Crítico (≥10K)',            pl.col('MONTO')>=10000),
    ('R03: Ráfaga 5min (≥3 trx)',            pl.col('FLAG_VEL_RAFAGA_5MIN')==1),
    ('R04: Velocidad Alta 24H',              pl.col('FLAG_VEL_ALTA_24H')==1),
    ('R05: Anomalía Monto |z|>2',            pl.col('FLAG_ANOMALIA_MONTO')==1),
    ('R06: Madrugada (0-5h)',                pl.col('FRANJA_HORARIA')=='Madrugada'),
    ('R07: Vaciado cuenta ≥80%',             pl.col('FLAG_VACIADO_CUENTA')==1),
    ('R08: Cuenta nueva (≤3 trx)',           pl.col('FLAG_CUENTA_NUEVA')==1),
    ('R09: Beneficiario nulo',               pl.col('FLAG_BENE_NULO')==1),
    ('R10: Dispositivo ATO',                 pl.col('FLAG_DISPOSITIVO_MULTIUSUARIO')==1),
    ('R11: Beneficiario concentrado',        pl.col('FLAG_BENE_CONCENTRADO')==1),
    ('R12: Anomalía + Ráfaga',               pl.col('FLAG_ANOMALIA_RAFAGA')==1),
    ('R13: Mula (Vaciado + Cta Nueva)',      pl.col('FLAG_MULA')==1),
    ('R14: ATO Nocturno',                    pl.col('FLAG_ATO_NOCTURNO')==1),
    ('R15: Vaciado + Madrugada',             (pl.col('FLAG_VACIADO_CUENTA')==1) & (pl.col('FRANJA_HORARIA')=='Madrugada')),
    ('R16: MOT + Madrugada + Vaciado',       (pl.col('CANAL')=='MOT') & (pl.col('FRANJA_HORARIA')=='Madrugada') & (pl.col('FLAG_VACIADO_CUENTA')==1)),
]

df_reglas = pd.DataFrame([evaluar_regla(n,m) for n,m in REGLAS]).sort_values('F1', ascending=False)
show(df_reglas, caption='⚖️ Reglas ordenadas por F1 (equilibrio Precision/Recall)')

In [ ]:
# Gráfico Precision vs Recall — quadrante ideal arriba a la derecha
fig = px.scatter(
    df_reglas, x='Recall', y='Precision', size='Monto_Cap', color='F1',
    text='Regla', hover_data=['Alertas','Fraudes_F','Normales_N','Pct_Volumen'],
    color_continuous_scale='Reds', size_max=50, height=600,
    title='Precision vs Recall — la mejor regla está arriba a la derecha')
fig.update_traces(textposition='top center', textfont=dict(size=9))
fig.update_layout(
    xaxis_title='Recall (% fraudes capturados)',
    yaxis_title='Precision (% alertas que son fraude)'
)
fig.show()

# Identificar la mejor regla
mejor = df_reglas.iloc[0]
print(f'\n🏆 REGLA CON MEJOR EQUILIBRIO: {mejor["Regla"]}')
print(f'   • Alertaríamos: {mejor["Alertas"]:,} transacciones ({mejor["Pct_Volumen"]}% del volumen)')
print(f'   • Capturaríamos: {mejor["Fraudes_F"]} fraudes ({mejor["Recall"]}% del total)')
print(f'   • Precisión: {mejor["Precision"]}% — de cada 100 alertas, ~{mejor["Precision"]:.0f} son fraude')
print(f'   • Monto capturado: S/{mejor["Monto_Cap"]:,.2f}')
print(f'   • Normales alertadas (ruido): {mejor["Normales_N"]:,}')

---
# 11. Árbol Multivariable — Modelo Destilado con LightGBM

**Estrategia:**
1. Entrenar LightGBM (modelo potente) con TODAS las variables
2. Usar las predicciones del LightGBM como pseudo-target
3. Destilar las reglas en un árbol simple e interpretable

Las reglas del árbol final se pueden implementar directamente en Monitor.

In [ ]:
import lightgbm as lgb
from sklearn.tree import DecisionTreeClassifier, export_text

# Variables predictoras
VARS_NUM = [c for c in [
    'MONTO','SALDO_DISPONIBLE','RATIO_MONTO_SALDO','CNT_MES_PREVIO',
    'ZSCORE_MONTO_CUENTA','N_CUENTAS_ORIGEN_AL_BENE','N_CUENTAS_POR_DISPOSITIVO',
    'VEL_TRX_5MIN','VEL_TRX_1H','VEL_TRX_24H','VEL_TRX_7D',
    'VEL_MONTO_5MIN','VEL_MONTO_1H','VEL_MONTO_24H','VEL_MONTO_7D',
    'HORA','DIA_SEMANA',
] if c in df.columns]

VARS_CAT = [c for c in ['CANAL','FRANJA_HORARIA'] if c in df.columns]

# Dataset solo clasificadas
df_model = df_cal.select(VARS_NUM + VARS_CAT + ['FLAG_FRAUDE']).to_pandas()

# One-hot de categóricas
df_model = pd.get_dummies(df_model, columns=VARS_CAT, drop_first=True)

# Quitar nulos
df_model = df_model.dropna(subset=['FLAG_FRAUDE'])
for c in df_model.columns:
    if df_model[c].dtype != 'object':
        df_model[c] = df_model[c].fillna(df_model[c].median())

X = df_model.drop('FLAG_FRAUDE', axis=1)
y = df_model['FLAG_FRAUDE'].astype(int)

# ── Paso 1: LightGBM ──────────────────────────────────────────────────────────
lgbm = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=5,
    is_unbalance=True, random_state=42, verbose=-1
)
lgbm.fit(X, y)
y_pred_lgbm = lgbm.predict_proba(X)[:, 1]

# Importancia de variables
imp = pd.DataFrame({
    'variable'   : X.columns,
    'importancia': lgbm.feature_importances_
}).sort_values('importancia', ascending=False).head(15)

fig = px.bar(imp, y='variable', x='importancia', orientation='h',
             color='importancia', color_continuous_scale='Reds',
             title='Top 15 variables — Importancia LightGBM',
             height=500)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

In [ ]:
# ── Paso 2: Destilar en árbol simple ─────────────────────────────────────────
y_destilado = (y_pred_lgbm > 0.5).astype(int)   # pseudo-target del LGBM

arbol_dest = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=20, class_weight='balanced'
)
arbol_dest.fit(X, y_destilado)

# Reglas en texto
reglas_texto = export_text(arbol_dest, feature_names=list(X.columns), max_depth=4)
print('📋 REGLAS DESTILADAS DEL ÁRBOL:\n')
print(reglas_texto)

# Visualización del árbol
fig_dest, ax = plt.subplots(figsize=(20, 8))
plot_tree(arbol_dest, feature_names=list(X.columns), class_names=['No Fraude','Fraude'],
          filled=True, rounded=True, fontsize=8, ax=ax,
          impurity=False, proportion=True)
ax.set_title('Árbol Destilado de LightGBM — Reglas listas para Monitor',
             fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

---
# 12. Export — Parquet para Power BI

In [ ]:
OUTPUT_DIR = BASE_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset completo con todos los flags y features de velocidad
df.write_parquet(OUTPUT_DIR / 'transferencias_terceros_FLAGS.parquet')

# Solo fraudes
(df_f.sort('MONTO', descending=True).to_pandas()
    .to_excel(OUTPUT_DIR/'fraudes.xlsx', index=False))

# Reglas
df_reglas.to_excel(OUTPUT_DIR/'evaluacion_reglas.xlsx', index=False)

# Top destinos
top_destino.to_excel(OUTPUT_DIR/'top_cuentas_destino.xlsx', index=False)

# Velocidad por cuenta
vel_cuenta.to_excel(OUTPUT_DIR/'velocidad_por_cuenta.xlsx', index=False)

print('✅ Archivos exportados:')
print(f'   → transferencias_terceros_FLAGS.parquet  ({df.shape[0]:,} filas × {df.shape[1]} cols)')
print(f'   → fraudes.xlsx ({df_f.shape[0]:,} filas)')
print(f'   → evaluacion_reglas.xlsx')
print(f'   → top_cuentas_destino.xlsx')
print(f'   → velocidad_por_cuenta.xlsx')
print(f'\n📂 Ruta: {OUTPUT_DIR}')

flags = sorted([c for c in df.columns if c.startswith('FLAG_')])
print(f'\n📊 Flags disponibles en Power BI ({len(flags)}):')
for f in flags: print(f'   · {f}')

vels = sorted([c for c in df.columns if c.startswith('VEL_')])
print(f'\n⚡ Variables de velocidad ({len(vels)}):')
for v in vels: print(f'   · {v}')

---
## 📝 Conclusiones

| # | Hallazgo | Sección |
|---|---|---|
| 1 | Madrugada concentra mayor LIFT de fraude | 5.3 |
| 2 | Canal MOT en madrugada = mayor riesgo | 5.2 |
| 3 | Cortes óptimos de monto los define el árbol CART | 4.4 |
| 4 | Vaciado de cuenta (≥80%) discrimina fraude | 11 |
| 5 | Cuentas mula: CC_K05 ≤3 + vaciado | 2.5 |
| 6 | ATO: dispositivos con ≥3 cuentas | 9 |
| 7 | Red de fraude: beneficiarios concentrados | 9 |
| 8 | Velocidad en ventanas cortas (5-10 min) más predictiva | 7 |

**Próximos pasos:**
1. Conectar `transferencias_terceros_FLAGS.parquet` a Power BI
2. Implementar la regla con mejor F1 (sección 10) en Monitor
3. Considerar el modelo destilado como sistema de scoring complementario

---
*Análisis generado automáticamente · Prevención de Fraude · Scotiabank Perú*